# Train the trajectory adapter on a Colab A100
Open this in VS Code, **Select Kernel -> Colab -> New Colab Server**, pick an A100, then run the cells in order.
Needs `ie643_colab.zip` (code + `data/dataset_got10k`, built locally) in **My Drive** root.
Checkpoints and logs go to `My Drive/ie643_ckpt/<RUN>` (set `RUN` in the training cell), so a disconnect loses nothing: re-run all cells and training resumes from that run's `last.pt`.
**Use a NEW `RUN` name whenever the config or code changes**, otherwise it resumes the old weights.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; assert torch.cuda.is_available(), "pick a GPU runtime"
assert torch.cuda.get_device_properties(0).total_memory > 14e9, "13B needs an A100 (40/80 GB), the 2B config (T4) needs >= 15 GB"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
if not os.path.isdir('/content/ie643'):
    !unzip -q /content/drive/MyDrive/ie643_colab.zip -d /content/ie643
%cd /content/ie643
!ls

In [ ]:
# Colab already has a matching torch; install the rest (transformers is pinned <5 in requirements.txt).
!grep -v -E '^(torch|gradio)' requirements.txt > /tmp/req.txt && pip install -q -r /tmp/req.txt

In [ ]:
# Optional: Hugging Face login (faster, more reliable model download). The token is typed, never printed or saved to a file.
import getpass
from huggingface_hub import login
tok = getpass.getpass('HF token (blank to skip): ')
if tok: login(tok)
del tok

In [ ]:
# Sanity check on the real data and config (a few seconds, no training).
!python -m pytest tests -q

In [ ]:
# TRAIN. Resumes automatically from Drive if a previous session saved last.pt.
import os
CFG = ''   # '' = 13B (A100) | '--config configs/ltx2b.yaml' = 2B, fast, A100 | '--config configs/t4.yaml' = 2B fp16 small frames, T4
RUN = 'run2_lr3e-5'   # <- new name for every new experiment (run1 diverged at lr 5e-4)
CK = f'/content/drive/MyDrive/ie643_ckpt/{RUN}'
os.makedirs(CK, exist_ok=True)
resume = f'--resume {CK}/last.pt' if os.path.exists(f'{CK}/last.pt') else ''
!python -u scripts/train.py {CFG} --save_dir {CK} {resume} 2>&1 | tee -a {CK}/train_stdout.log

In [ ]:
# Did the adapter learn to use the trajectory?  cond_gain > 0 and inject_ratio well above 0 = yes.
import json
for l in open(f'{CK}/log.jsonl'):
    r = json.loads(l)
    print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.items()
           if k in ('epoch','train_loss','val_loss','cond_gain','train_cond_gain','inject_ratio','adapter_scale')})

In [ ]:
# Benchmark: adapter vs plain LTX, tracked with CoTracker3 (ATE, direction, progress).
!python scripts/benchmark.py {CFG} --checkpoint {CK}/best.pt --num_images 8 --out {CK}/benchmark